# MuSeg-AI Thigh Segmentation — Asian Dataset (Lambda)

Runs [fabianbalsiger/museg-ai](https://github.com/fabianbalsiger/museg-ai) (`thigh-model3`)
on the **MRI_data_asian** dataset — **Thigh only** — in two modes:

| Mode | Input | Output folder |
|---|---|---|
| Water-only | `Water.nii.gz` as both channels | `~/museg_asian_segs/water_only/` |
| Dixon-based | in-phase = Water+Fat, out-phase = Water−Fat | `~/museg_asian_segs/dixon_based/` |

Data structure: `MRI_data_asian/MRI_data/{01-25}/Thigh/{Water,Fat}.nii.gz`

## 1 — Upload data to Lambda
```bash
rsync -avz -e "ssh -i /tmp/lambda_key -o StrictHostKeyChecking=no" \
  /tmp/docker-desktop-root/run/desktop/mnt/host/c/Projects/dissector/eval_notebooks/MRI_data_asian \
  ubuntu@<YOUR-LAMBDA-IP>:~/
```

## 2 — Download results when done
```bash
# Water-only
rsync -avz -e "ssh -i /tmp/lambda_key -o StrictHostKeyChecking=no" \
  ubuntu@<YOUR-LAMBDA-IP>:~/museg_asian_segs/water_only/ \
  /tmp/docker-desktop-root/run/desktop/mnt/host/c/Projects/dissector/eval_notebooks/museg/asian_segs/water_only/

# Dixon-based
rsync -avz -e "ssh -i /tmp/lambda_key -o StrictHostKeyChecking=no" \
  ubuntu@<YOUR-LAMBDA-IP>:~/museg_asian_segs/dixon_based/ \
  /tmp/docker-desktop-root/run/desktop/mnt/host/c/Projects/dissector/eval_notebooks/museg/asian_segs/dixon_based/
```

**Terminate the instance when done.**

In [ ]:
import subprocess, sys

def sh(cmd):
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    out = (r.stdout + r.stderr).strip()
    if r.returncode != 0:
        print(f'[WARN] {cmd[:80]}: {out[:300]}')
    else:
        print(f'OK: {cmd[:60]}')
    return r.returncode == 0

# ── Docker ───────────────────────────────────────────────────────────────────
r = subprocess.run('which docker', shell=True, capture_output=True)
if r.returncode != 0:
    print('Installing docker.io ...')
    sh('sudo apt-get update -qq')
    sh('sudo apt-get install -y docker.io')
else:
    print('Docker already present:', r.stdout.strip())

sh('sudo systemctl start docker')
sh('sudo chmod 666 /var/run/docker.sock')

# ── museg-ai ─────────────────────────────────────────────────────────────────
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'git+https://github.com/fabianbalsiger/museg-ai.git'])

import importlib; importlib.invalidate_caches()
import musegai
print('museg-ai version:', musegai.__version__)

In [ ]:
import docker
try:
    client = docker.from_env()
    client.ping()
    print('Docker is running.')
except Exception as e:
    raise RuntimeError(
        'Docker is not reachable. Re-run the setup cell, then retry.\n'
        f'Original error: {e}'
    )

In [ ]:
import glob, os
import numpy as np
from musegai import api
from musegai.api import Volume

DATA_ROOT   = os.path.expanduser('~/MRI_data_asian/MRI_data')
OUTPUT_BASE = os.path.expanduser('~/museg_asian_segs')

OUTPUT_WATER_ONLY  = os.path.join(OUTPUT_BASE, 'water_only')
OUTPUT_DIXON_BASED = os.path.join(OUTPUT_BASE, 'dixon_based')

os.makedirs(OUTPUT_WATER_ONLY,  exist_ok=True)
os.makedirs(OUTPUT_DIXON_BASED, exist_ok=True)

LABEL_MAP = {
    1:  'Vastus_Lateralis',
    2:  'Vastus_Intermedius',
    3:  'Vastus_Medialis',
    4:  'Rectus_Femoris',
    5:  'Sartorius',
    6:  'Gracilis',
    7:  'Semimembranosus',
    8:  'Semitendinosus',
    9:  'Biceps_Femoris',
    10: 'Biceps_Femoris_Short',
    11: 'Adductor_Magnus',
    12: 'Adductor_Longus',
    13: 'Adductor_Brevis',
}

# Discover jobs: subjects that have both Water and Fat
jobs = []
for subject in sorted(os.listdir(DATA_ROOT)):
    water_path = os.path.join(DATA_ROOT, subject, 'Thigh', 'Water.nii.gz')
    fat_path   = os.path.join(DATA_ROOT, subject, 'Thigh', 'Fat.nii.gz')
    if os.path.exists(water_path) and os.path.exists(fat_path):
        jobs.append((subject, water_path, fat_path))
    elif os.path.exists(water_path):
        jobs.append((subject, water_path, None))
        print(f'  [WARN] {subject}: Fat.nii.gz not found — water-only mode only')

print(f'Found {len(jobs)} subjects')
for subj, w, f in jobs:
    print(f'  {subj}  water={os.path.exists(w)}  fat={f is not None and os.path.exists(f)}')

## Water-only segmentation
Water volume is passed as both the in-phase and out-of-phase channel.

In [ ]:
for subject, water_path, fat_path in jobs:
    out_subdir = os.path.join(OUTPUT_WATER_ONLY, subject, 'Thigh')
    out_path   = os.path.join(out_subdir, 'Thigh_seg.nii.gz')

    if os.path.exists(out_path):
        print(f'Skipping (done): {subject} water-only')
        continue

    print(f'\nProcessing (water-only): {subject}')
    os.makedirs(out_subdir, exist_ok=True)

    water_vol = Volume.load(water_path)
    print(f'  Shape: {water_vol.shape}  Spacing: {water_vol.spacing}')

    results, labels = api.segment_volumes(
        {subject: [water_vol, water_vol]},
        model='thigh-model3',
        side='left+right',
    )

    segmentation = results[subject]
    segmentation.save(out_path)
    print(f'  Saved -> {out_path}')

    seg_arr = segmentation.array
    print(f'  Labels present: {sorted(np.unique(seg_arr).tolist())}')
    print(f'  {"Label":<6} {"Muscle":<25} {"Voxels":>10}')
    for idx, name in LABEL_MAP.items():
        n = int((seg_arr == idx).sum())
        if n > 0:
            print(f'  {idx:<6} {name:<25} {n:>10,}')

print('\nWater-only done.')

## Dixon-based segmentation
In-phase = Water + Fat; out-of-phase = Water − Fat.

In [ ]:
for subject, water_path, fat_path in jobs:
    if fat_path is None:
        print(f'Skipping (no Fat): {subject}')
        continue

    out_subdir = os.path.join(OUTPUT_DIXON_BASED, subject, 'Thigh')
    out_path   = os.path.join(out_subdir, 'Thigh_seg.nii.gz')

    if os.path.exists(out_path):
        print(f'Skipping (done): {subject} dixon-based')
        continue

    print(f'\nProcessing (dixon-based): {subject}')
    os.makedirs(out_subdir, exist_ok=True)

    water_vol = Volume.load(water_path)
    fat_vol   = Volume.load(fat_path)
    print(f'  Shape: {water_vol.shape}  Spacing: {water_vol.spacing}')

    inphase_arr  = water_vol.array.astype(float) + fat_vol.array.astype(float)
    outphase_arr = water_vol.array.astype(float) - fat_vol.array.astype(float)

    inphase_vol  = Volume(inphase_arr,  **water_vol.metadata)
    outphase_vol = Volume(outphase_arr, **water_vol.metadata)

    results, labels = api.segment_volumes(
        {subject: [inphase_vol, outphase_vol]},
        model='thigh-model3',
        side='left+right',
    )

    segmentation = results[subject]
    segmentation.save(out_path)
    print(f'  Saved -> {out_path}')

    seg_arr = segmentation.array
    print(f'  Labels present: {sorted(np.unique(seg_arr).tolist())}')
    print(f'  {"Label":<6} {"Muscle":<25} {"Voxels":>10}')
    for idx, name in LABEL_MAP.items():
        n = int((seg_arr == idx).sum())
        if n > 0:
            print(f'  {idx:<6} {name:<25} {n:>10,}')

print('\nDixon-based done.')

In [ ]:
# ── Sanity check ─────────────────────────────────────────────────────────────
for mode, out_dir in [('water_only', OUTPUT_WATER_ONLY), ('dixon_based', OUTPUT_DIXON_BASED)]:
    results = sorted(glob.glob(os.path.join(out_dir, '*', 'Thigh', 'Thigh_seg.nii.gz')))
    print(f'{mode}: {len(results)} / {len(jobs)} outputs')
    if results:
        sample = Volume.load(results[0])
        print(f'  Sample : {results[0]}')
        print(f'  Shape  : {sample.shape}')
        print(f'  Labels : {sorted(np.unique(sample.array).tolist())}')